# EfficientNetV2S 2.0 — Dual-Head (Classification + Nutrition)

**Changes from v1.1 (Food-101):**
- Dataset: Food-101 (101 classes) → MM-Food-100K + Fruits-360 (385 classes)
- Model output: single classification head → dual head (class + nutrition regression)
- Labels loaded from `label_map.json` instead of `classes.txt`
- tf.data pipeline reads from scanned directory + CSV nutrition lookup
- Loss: multi-task (CategoricalCrossentropy + Huber)
- Fixed `random_crop` bug from v1.1 (removed batch dim from size)

**Kept from v1.1:**
- Conv head architecture (1×1 bottleneck → 3×3 spatial → MaxPool → GAP)
- Phase 1 / Phase 2 training strategy
- EpochTimer, save_model_info, training curves, confusion matrix
- TFLite export

In [1]:
!pip install tensorflow keras scikit-learn seaborn -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 1. Imports & GPU Setup

In [2]:
import os
import time
import json
import inspect
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

import tensorflow as tf
import keras
from keras import layers, Model, regularizers
from keras.applications import EfficientNetV2S
from keras.applications.efficientnet_v2 import preprocess_input
from keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

# -- GPU setup (same as v1.1) --
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

# -- Mixed precision: ~30% faster on RTX 3060 Ti, halves activation memory --
tf.keras.mixed_precision.set_global_policy('mixed_float16')

!nvidia-smi
print(f'TF  : {tf.__version__}') 
print(f'GPU : {tf.config.list_physical_devices("GPU")}')
print(f'Policy: {tf.keras.mixed_precision.global_policy()}')

2026-05-26 20:48:33.001154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779828513.017064   21001 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779828513.021967   21001 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-26 20:48:33.039872: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Tue May 26 20:48:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.52.01              Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 Ti     On  |   00000000:06:00.0  On |                  N/A |
|  0%   43C    P8             17W /  220W |     778MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Config

In [3]:
MODEL_NAME = 'EfficientNetV2S_2.0'
BASE_DIR   = '/tf/data/models'
SAVE_DIR   = f'{BASE_DIR}/{MODEL_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)

# -- Data paths --
DATA_ROOT    = '/tf/data/food-mm'       # root of downloaded images
TRAIN_DIR    = f'{DATA_ROOT}/train'
VAL_DIR      = f'{DATA_ROOT}/val'
LABEL_MAP    = '/tf/data/output/label_map.json'
TRAIN_CSV    = '/tf/data/output/train.csv'
VAL_CSV      = '/tf/data/output/val.csv'
NUT_STATS    = '/tf/data/output/nutrition_stats.json'

# -- Training --
IMG_SIZE          = (384, 384)
BATCH_SIZE        = 32
FINE_TUNE_BATCH_SIZE = 16
EPOCHS            = 60
FINE_TUNE_EPOCHS  = 20
AUTOTUNE          = tf.data.AUTOTUNE

# -- Multi-task loss weights --
CLS_LOSS_WEIGHT   = 1.0    # classification
NUT_LOSS_WEIGHT   = 0.5    # nutrition regression

# -- Load label map --
with open(LABEL_MAP) as f:
    idx_to_class = json.load(f)           # {'0': 'Beef Noodles', ...}

N_CLASSES    = len(idx_to_class)
N_class      = {int(k): v for k, v in idx_to_class.items()}
class_N      = {v: k for k, v in N_class.items()}

print(f'N_CLASSES : {N_CLASSES}')
print(f'Sample    : {list(N_class.items())[:5]}')

N_CLASSES : 1407
Sample    : [(0, 'Apple'), (1, 'Apple slices'), (2, 'Apples'), (3, 'Apricot'), (4, 'Apricots')]


## 3. Nutrition Normalization Stats

In [4]:
with open(NUT_STATS) as f:
    nut_stats = json.load(f)

NUT_COLS = ['calories', 'protein', 'fat', 'carbs']

NUT_MIN  = np.array([nut_stats[c]['min']  for c in NUT_COLS], dtype=np.float32)
NUT_MAX  = np.array([nut_stats[c]['max']  for c in NUT_COLS], dtype=np.float32)

def normalize_nutrition(values):
    """Scale raw nutrition values to [0, 1] using training min/max."""
    return (values - NUT_MIN) / (NUT_MAX - NUT_MIN + 1e-8)

def denormalize_nutrition(values):
    """Invert normalization — use at inference time."""
    return values * (NUT_MAX - NUT_MIN + 1e-8) + NUT_MIN

print('Nutrition normalization ranges:')
for col, mn, mx in zip(NUT_COLS, NUT_MIN, NUT_MAX):
    print(f'  {col:<10}  min={mn:.1f}  max={mx:.1f}')

Nutrition normalization ranges:
  calories    min=2.0  max=3000.0
  protein     min=0.0  max=250.0
  fat         min=0.0  max=250.0
  carbs       min=0.0  max=400.0


## 4. Build Nutrition Lookup (median per class)

In [5]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

# Compute median nutrition per label from training set
nut_by_label = (
    train_df.groupby('label')[NUT_COLS]
    .median()
    .reset_index()
)

# Dict: label_int -> normalized np.array([cal, protein, fat, carbs])
label_to_nutrition = {}
for _, row in nut_by_label.iterrows():
    raw = np.array([row[c] for c in NUT_COLS], dtype=np.float32)
    label_to_nutrition[int(row['label'])] = normalize_nutrition(raw)

# Fallback for any label not in CSV (shouldn't happen, but safety)
FALLBACK_NUT = normalize_nutrition(np.array([60., 1., 0.3, 15.], dtype=np.float32))

print(f'Nutrition lookup built for {len(label_to_nutrition)} labels')
print('Sample:')
for lbl in list(label_to_nutrition.keys())[:3]:
    print(f'  Label {lbl} ({N_class[lbl]}): {label_to_nutrition[lbl].round(3)}')

Nutrition lookup built for 1407 labels
Sample:
  Label 0 (Apple): [0.017 0.001 0.001 0.035]
  Label 1 (Apple slices): [0.017 0.001 0.001 0.035]
  Label 2 (Apples): [0.173 0.004 0.004 0.35 ]


## 5. tf.data Pipeline

In [6]:
# -- Scan directory -> list of (path, label, nutrition_normalized) --
def scan_directory(split_dir, label_to_nutrition, fallback):
    paths, labels, nutritions = [], [], []
    for label_folder in sorted(os.listdir(split_dir), key=lambda x: int(x)):
        label     = int(label_folder)
        folder    = os.path.join(split_dir, label_folder)
        nutrition = label_to_nutrition.get(label, fallback)
        for fname in os.listdir(folder):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                paths.append(os.path.join(folder, fname))
                labels.append(label)
                nutritions.append(nutrition)
    return paths, labels, nutritions

print('Scanning train...')
train_paths, train_labels, train_nuts = scan_directory(TRAIN_DIR, label_to_nutrition, FALLBACK_NUT)
print(f'  {len(train_paths):,} train images')

print('Scanning val...')
val_paths, val_labels, val_nuts = scan_directory(VAL_DIR, label_to_nutrition, FALLBACK_NUT)
print(f'  {len(val_paths):,} val images')

Scanning train...
  192,289 train images
Scanning val...
  59,492 val images


In [7]:
from collections import Counter

# -- Compute per-class sample counts for tier assignment --
label_counts = Counter(train_labels)

tier0 = sum(1 for l in train_labels if label_counts[l] >= 500)
tier1 = sum(1 for l in train_labels if 100 <= label_counts[l] < 500)
tier2 = sum(1 for l in train_labels if label_counts[l] < 100)
print(f'Tier 0 (light  aug, >=500 samples) : {tier0:,} images')
print(f'Tier 1 (medium aug, 100-499)       : {tier1:,} images')
print(f'Tier 2 (heavy  aug, <100 samples)  : {tier2:,} images')

# -- Image loader --
@tf.function
def load_image(path, label, nutrition):
    raw   = tf.io.read_file(path)
    image = tf.image.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label, nutrition

# -- Three augmentation levels --
@tf.function
def augment_light(image, label, nutrition):
    """Dominant classes (Food-101 heavy) — flip only to preserve clean signal."""
    image = tf.image.random_flip_left_right(image)
    return image, label, nutrition

@tf.function
def augment_medium(image, label, nutrition):
    """Medium classes — standard augmentation."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.05)
    pad_h = int(IMG_SIZE[0] * 0.1)
    pad_w = int(IMG_SIZE[1] * 0.1)
    image = tf.image.pad_to_bounding_box(
        image, pad_h, pad_w,
        IMG_SIZE[0] + 2 * pad_h, IMG_SIZE[1] + 2 * pad_w
    )
    image = tf.image.random_crop(image, size=[IMG_SIZE[0], IMG_SIZE[1], 3])
    return image, label, nutrition

@tf.function
def augment_heavy(image, label, nutrition):
    """Rare classes — aggressive augmentation to compensate low sample count."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.6, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.4)
    image = tf.image.random_hue(image, max_delta=0.1)
    pad_h = int(IMG_SIZE[0] * 0.15)
    pad_w = int(IMG_SIZE[1] * 0.15)
    image = tf.image.pad_to_bounding_box(
        image, pad_h, pad_w,
        IMG_SIZE[0] + 2 * pad_h, IMG_SIZE[1] + 2 * pad_w
    )
    image = tf.image.random_crop(image, size=[IMG_SIZE[0], IMG_SIZE[1], 3])
    return image, label, nutrition

@tf.function
def format_sample(image, label, nutrition):
    image    = preprocess_input(image)
    label_oh = tf.one_hot(label, N_CLASSES)
    return image, {'class_output': label_oh, 'nutrition_output': nutrition}

def make_tier_ds(paths, labels, nutritions, aug_fn):
    ds = tf.data.Dataset.from_tensor_slices((
        tf.constant(paths),
        tf.constant(labels, dtype=tf.int32),
        tf.constant(np.array(nutritions), dtype=tf.float32)
    ))
    ds = ds.shuffle(min(len(paths), 5000), reshuffle_each_iteration=True)
    ds = ds.map(load_image,    num_parallel_calls=AUTOTUNE)
    ds = ds.map(aug_fn,        num_parallel_calls=AUTOTUNE)
    ds = ds.map(format_sample, num_parallel_calls=AUTOTUNE)
    return ds

def build_tiered_dataset(paths, labels, nutritions):
    """Split into 3 aug tiers, sample_from_datasets with equal weight per tier."""
    t0_p, t0_l, t0_n = [], [], []
    t1_p, t1_l, t1_n = [], [], []
    t2_p, t2_l, t2_n = [], [], []

    for p, l, n in zip(paths, labels, nutritions):
        count = label_counts[l]
        if count >= 500:
            t0_p.append(p); t0_l.append(l); t0_n.append(n)
        elif count >= 100:
            t1_p.append(p); t1_l.append(l); t1_n.append(n)
        else:
            t2_p.append(p); t2_l.append(l); t2_n.append(n)

    print(f'  Tier 0 (light)  : {len(t0_p):,} samples')
    print(f'  Tier 1 (medium) : {len(t1_p):,} samples')
    print(f'  Tier 2 (heavy)  : {len(t2_p):,} samples')

    ds0 = make_tier_ds(t0_p, t0_l, t0_n, augment_light)
    ds1 = make_tier_ds(t1_p, t1_l, t1_n, augment_medium)
    ds2 = make_tier_ds(t2_p, t2_l, t2_n, augment_heavy)

    n0, n1, n2 = len(t0_p), len(t1_p), len(t2_p)
    total = n0 + n1 + n2

    # Equal weight per tier so rare classes get proportionally more gradient updates
    combined = tf.data.Dataset.sample_from_datasets(
        [ds0, ds1, ds2],
        weights=[1/3, 1/3, 1/3]
    )
    return combined.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def build_val_dataset(paths, labels, nutritions):
    ds = tf.data.Dataset.from_tensor_slices((
        tf.constant(paths),
        tf.constant(labels, dtype=tf.int32),
        tf.constant(np.array(nutritions), dtype=tf.float32)
    ))
    ds = ds.map(load_image,    num_parallel_calls=AUTOTUNE)
    ds = ds.map(format_sample, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print('Building train dataset (tiered)...')
train_ds = build_tiered_dataset(train_paths, train_labels, train_nuts)
print('Building val dataset...')
val_ds   = build_val_dataset(val_paths, val_labels, val_nuts)

# RAM budget cap
options = tf.data.Options()
options.autotune.ram_budget = 2 * 1024 * 1024 * 1024
train_ds = train_ds.with_options(options)
val_ds   = val_ds.with_options(options)

# Verify shapes
for images, targets in train_ds.take(1):
    print(f'Image batch     : {images.shape}  dtype={images.dtype}')
    print(f'Class labels    : {targets["class_output"].shape}')
    print(f'Nutrition labels: {targets["nutrition_output"].shape}')

I0000 00:00:1779828515.345900   21001 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:06:00.0, compute capability: 8.6


Image batch    : (32, 384, 384, 3)  dtype=<dtype: 'float32'>
Class labels   : (32, 1407)
Nutrition labels: (32, 4)


2026-05-26 20:48:36.072667: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 6. Helpers (from v1.1)

In [8]:
class EpochTimer(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.time()
    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self._t
        print(f'  Epoch {epoch+1} time: {elapsed/60:.1f} min  ({elapsed:.0f}s)')


def _layer_args(layer):
    cfg = layer.get_config()
    if 'units' in cfg:
        s = str(cfg['units'])
        if cfg.get('activation'): s += f", {cfg['activation']}"
        if cfg.get('kernel_regularizer'): s += ', l2'
        return s
    if 'rate' in cfg: return str(cfg['rate'])
    if 'axis' in cfg: return f"axis={cfg['axis']}"
    return ''


def save_model_info(model, save_dir):
    import datetime
    optimizer  = model.optimizer
    base_model = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)
    opt_name   = optimizer.__class__.__name__
    try:
        lr     = optimizer.learning_rate
        lr_str = f'{float(lr):.2e}' if hasattr(lr, '__float__') else lr.__class__.__name__
    except:
        lr_str = 'unknown'

    base_name         = base_model.name if base_model else 'none'
    total_layers      = len(base_model.layers) if base_model else 0
    frozen_layers     = sum(1 for l in (base_model.layers if base_model else []) if not l.trainable)
    trainable_layers  = total_layers - frozen_layers

    try:
        aug_src = inspect.getsource(augment)
        aug_ops = [l.strip() for l in aug_src.splitlines()
                   if 'tf.image.' in l and not l.strip().startswith('#')]
    except:
        aug_ops = ['(could not extract)']

    info_path = f'{save_dir}/{MODEL_NAME}_model_info.txt'
    with open(info_path, 'w') as f:
        f.write(f'DATE              : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
        f.write(f'MODEL_NAME        : {MODEL_NAME}\n')
        f.write(f'VERSION           : 2.0 (dual-head: classification + nutrition)\n')
        f.write(f'EPOCHS            : {EPOCHS}\n')
        f.write(f'FINE_TUNE_EPOCHS  : {FINE_TUNE_EPOCHS}\n')
        f.write(f'IMG_SIZE          : {IMG_SIZE}\n')
        f.write(f'BATCH_SIZE        : {BATCH_SIZE}\n')
        f.write(f'N_CLASSES         : {N_CLASSES}\n')
        f.write(f'CLS_LOSS_WEIGHT   : {CLS_LOSS_WEIGHT}\n')
        f.write(f'NUT_LOSS_WEIGHT   : {NUT_LOSS_WEIGHT}\n')
        f.write(f'\n-- Optimizer --\n')
        f.write(f'  class           : {opt_name}\n')
        f.write(f'  learning_rate   : {lr_str}\n')
        f.write(f'\n-- Base Model --\n')
        f.write(f'  name            : {base_name}\n')
        f.write(f'  total layers    : {total_layers}\n')
        f.write(f'  frozen layers   : {frozen_layers}\n')
        f.write(f'  trainable layers: {trainable_layers}\n')
        f.write(f'\n-- Augmentation ops --\n')
        for op in aug_ops:
            f.write(f'  {op}\n')
        f.write(f'\n-- Full Layer Summary --\n')
        model.summary(print_fn=lambda line: f.write(line + '\n'), show_trainable=True)
    print(f'model_info saved → {info_path}')


class LivePlotCallback(tf.keras.callbacks.Callback):
    """Plots train/val loss and accuracy after every epoch inline."""

    def __init__(self, save_dir, model_name, phase='phase1'):
        super().__init__()
        self.save_dir   = save_dir
        self.model_name = model_name
        self.phase      = phase
        self._hist      = {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for k, v in logs.items():
            self._hist.setdefault(k, []).append(float(v))

        epochs = list(range(1, epoch + 2))
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(
            f'{self.model_name} — {self.phase}  |  epoch {epoch + 1}',
            fontsize=12
        )

        def _plot(ax, train_key, val_key, title, fmt='{:.4f}'):
            tr = self._hist.get(train_key, [])
            vl = self._hist.get(val_key,   [])
            if tr:
                ax.plot(epochs[:len(tr)], tr, 'b-o', markersize=3, label=f'Train ({fmt.format(tr[-1])})')
            if vl:
                ax.plot(epochs[:len(vl)], vl, 'r-o', markersize=3, label=f'Val   ({fmt.format(vl[-1])})')
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

        _plot(axes[0], 'loss',                     'val_loss',                     'Total Loss')
        _plot(axes[1], 'class_output_accuracy',    'val_class_output_accuracy',    'Top-1 Accuracy',   '{:.2%}')
        _plot(axes[2], 'nutrition_output_mae',     'val_nutrition_output_mae',     'Nutrition MAE')

        plt.tight_layout()
        path = f'{self.save_dir}/{self.model_name}_{self.phase}_live.png'
        plt.savefig(path, dpi=100, bbox_inches='tight')
        plt.show()
        plt.close(fig)

## 7. Build Model — Dual Head

In [9]:
# -- Backbone (same as v1.1) --
base_model = EfficientNetV2S(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)

# -- Conv head (kept from v1.1) --
x = layers.Conv2D(512, (1, 1), padding='same', use_bias=False,
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.SpatialDropout2D(0.3)(x)

x = layers.Conv2D(512, (3, 3), padding='same', use_bias=False,
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.SpatialDropout2D(0.3)(x)

x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

# -- Shared dense --
shared = layers.Dense(512, activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
shared = layers.Dropout(0.4)(shared)

# -- Classification head --
cls_x   = layers.Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(shared)
cls_x   = layers.Dropout(0.3)(cls_x)
# Cast to float32 before softmax (required for mixed precision stability)
cls_x   = layers.Activation('linear', dtype='float32')(cls_x)
cls_out = layers.Dense(N_CLASSES, activation='softmax',
                        dtype='float32', name='class_output')(cls_x)

# -- Nutrition regression head --
reg_x   = layers.Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(shared)
reg_x   = layers.Dropout(0.2)(reg_x)
# Sigmoid output: nutrition is normalized to [0,1]
reg_x   = layers.Activation('linear', dtype='float32')(reg_x)
reg_out = layers.Dense(4, activation='sigmoid',
                        dtype='float32', name='nutrition_output')(reg_x)

model = Model(inputs, [cls_out, reg_out])
model.summary(show_trainable=True)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)      ┃ Output Shape    ┃   Param # ┃ Connected to   ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1     │ (None, 384,     │         0 │ -              │   -   │
│ (InputLayer)      │ 384, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ efficientnetv2-s  │ (None, 12, 12,  │ 20,331,3… │ input_layer_1… │   N   │
│ (Functional)      │ 1280)           │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d (Conv2D)   │ (None, 12, 12,  │   655,360 │ efficientnetv… │   Y   │
│                   │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 12, 12,  │     2,048 │ conv2d[0][0]   │   Y   │
│ (BatchNormalizat… │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation        │ (None, 12, 12,  │         0 │ batch_normali… │   -   │
│ (Activation)      │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ spatial_dropout2d │ (None, 12, 12,  │         0 │ activation[0]… │   -   │
│ (SpatialDropout2… │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ conv2d_1 (Conv2D) │ (None, 12, 12,  │ 2,359,296 │ spatial_dropo… │   Y   │
│                   │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 12, 12,  │     2,048 │ conv2d_1[0][0] │   Y   │
│ (BatchNormalizat… │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ activation_1      │ (None, 12, 12,  │         0 │ batch_normali… │   -   │
│ (Activation)      │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ spatial_dropout2… │ (None, 12, 12,  │         0 │ activation_1[… │   -   │
│ (SpatialDropout2… │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ max_pooling2d     │ (None, 6, 6,    │         0 │ spatial_dropo… │   -   │
│ (MaxPooling2D)    │ 512)            │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ global_average_p… │ (None, 512)     │         0 │ max_pooling2d… │   -   │
│ (GlobalAveragePo… │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ batch_normalizat… │ (None, 512)     │     2,048 │ global_averag… │   Y   │
│ (BatchNormalizat… │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ dense (Dense)     │ (None, 512)     │   262,656 │ batch_normali… │   Y   │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ dropout (Dropout) │ (None, 512)     │         0 │ dense[0][0]    │   -   │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ dense_1 (Dense)   │ (None, 256)     │   131,328 │ dropout[0][0]  │   Y   │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ dense_2 (Dense)   │ (None, 256)     │   131,328 │ dropout[0][0]  │   Y   │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ dropout_1         │ (None, 256)     │         0 │ dense_1[0][0]  │   -   │
│ (Dropout)         │                 │           │                │     

 Total params: 24,240,099 (92.47 MB)

 Trainable params: 3,905,667 (14.90 MB)

 Non-trainable params: 20,334,432 (77.57 MB)

## 8. Phase 1 — Head Training

In [10]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'class_output':     keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        'nutrition_output': keras.losses.Huber(delta=0.1)
    },
    loss_weights={
        'class_output':     CLS_LOSS_WEIGHT,
        'nutrition_output': NUT_LOSS_WEIGHT
    },
    metrics={
        'class_output':     [
            'accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
        ],
        'nutrition_output': ['mae']
    }
)

save_model_info(model, SAVE_DIR)

from sklearn.utils.class_weight import compute_class_weight

all_labels  = np.array(train_labels)
cls_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(N_CLASSES),
    y=all_labels
)
sample_weights = np.array([cls_weights[l] for l in train_labels], dtype=np.float32)

print(f'Class weight range: {cls_weights.min():.3f} – {cls_weights.max():.3f}')
print(f'Median weight     : {np.median(cls_weights):.3f}')

def build_weighted_dataset(paths, labels, nutritions, weights):
    paths_t   = tf.constant(paths)
    labels_t  = tf.constant(labels, dtype=tf.int32)
    nuts_t    = tf.constant(np.array(nutritions), dtype=tf.float32)
    weights_t = tf.constant(weights, dtype=tf.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths_t, labels_t, nuts_t, weights_t))
    ds = ds.shuffle(8000, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l, n, w: (*load_image(p, l, n), w),        num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda img, l, n, w: (*augment(img, l, n), w),       num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda img, l, n, w: (*format_sample(img, l, n), w), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = build_weighted_dataset(train_paths, train_labels, train_nuts, sample_weights)

print('\n=== Phase 1: Head training ===')
ft_start = time.time()

history_ht = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(
            f'{SAVE_DIR}/{MODEL_NAME}_best_base.keras',
            save_best_only=True,
            monitor='val_class_output_accuracy'
        ),
        EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor='val_class_output_accuracy',
            mode='max'
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6
        ),
        CSVLogger(f'{SAVE_DIR}/{MODEL_NAME}_history_base.log'),
        EpochTimer(),
    ]
)

ft_total = time.time() - ft_start
h        = history_ht.history

print(f'\nPhase 1 total: {ft_total/3600:.2f}h  ({ft_total/60:.1f}min)')
print(f'  Train cls acc (last): {h["class_output_accuracy"][-1]*100:.2f}%')
print(f'  Val   cls acc (last): {h["val_class_output_accuracy"][-1]*100:.2f}%')
print(f'  Best  val cls acc   : {max(h["val_class_output_accuracy"])*100:.2f}%')
print(f'  Val   nut MAE (last): {h["val_nutrition_output_mae"][-1]:.4f}')

info_path = f'{SAVE_DIR}/{MODEL_NAME}_model_info.txt'
with open(info_path, 'a') as f:
    f.write(f'\n-- Phase 1 Results --\n')
    f.write(f'  Training time          : {ft_total/3600:.2f}h\n')
    f.write(f'  Train cls acc (last)   : {h["class_output_accuracy"][-1]*100:.2f}%\n')
    f.write(f'  Val   cls acc (last)   : {h["val_class_output_accuracy"][-1]*100:.2f}%\n')
    f.write(f'  Best  val cls acc      : {max(h["val_class_output_accuracy"])*100:.2f}%\n')
    f.write(f'  Val nut MAE (last)     : {h["val_nutrition_output_mae"][-1]:.4f}\n')

model.save(f'{SAVE_DIR}/{MODEL_NAME}_final_base.keras')
print('Saved phase 1 model.')


class LivePlotCallback(tf.keras.callbacks.Callback):
    """Plots train/val loss and accuracy after every epoch inline."""

    def __init__(self, save_dir, model_name, phase='phase1'):
        super().__init__()
        self.save_dir   = save_dir
        self.model_name = model_name
        self.phase      = phase
        self._hist      = {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for k, v in logs.items():
            self._hist.setdefault(k, []).append(float(v))

        epochs = list(range(1, epoch + 2))
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(
            f'{self.model_name} — {self.phase}  |  epoch {epoch + 1}',
            fontsize=12
        )

        def _plot(ax, train_key, val_key, title, fmt='{:.4f}'):
            tr = self._hist.get(train_key, [])
            vl = self._hist.get(val_key,   [])
            if tr:
                ax.plot(epochs[:len(tr)], tr, 'b-o', markersize=3, label=f'Train ({fmt.format(tr[-1])})')
            if vl:
                ax.plot(epochs[:len(vl)], vl, 'r-o', markersize=3, label=f'Val   ({fmt.format(vl[-1])})')
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

        _plot(axes[0], 'loss',                     'val_loss',                     'Total Loss')
        _plot(axes[1], 'class_output_accuracy',    'val_class_output_accuracy',    'Top-1 Accuracy',   '{:.2%}')
        _plot(axes[2], 'nutrition_output_mae',     'val_nutrition_output_mae',     'Nutrition MAE')

        plt.tight_layout()
        path = f'{self.save_dir}/{self.model_name}_{self.phase}_live.png'
        plt.savefig(path, dpi=100, bbox_inches='tight')
        plt.show()
        plt.close(fig)

model_info saved → /tf/data/models/EfficientNetV2S_2.0/EfficientNetV2S_2.0_model_info.txt
Class weight range: 0.008 – 17.083
Median weight     : 7.593

=== Phase 1: Head training ===
Epoch 1/60


I0000 00:00:1779828538.414063   21076 service.cc:148] XLA service 0x73cb6c41f6b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779828538.415032   21076 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Ti, Compute Capability 8.6
2026-05-26 20:48:59.148521: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779828542.483130   21076 cuda_dnn.cc:529] Loaded cuDNN version 92000
2026-05-26 20:49:06.235136: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_26387', 8 bytes spill stores, 8 bytes spill loads

2026-05-26 20:49:08.047798: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_28048', 376 b

6010/6010 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - class_output_accuracy: 0.3671 - class_output_loss: 6.8612 - class_output_top5_acc: 0.5815 - loss: 7.1051 - nutrition_output_loss: 0.0010 - nutrition_output_mae: 0.0432

2026-05-26 20:58:44.119847: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 8 bytes spill stores, 8 bytes spill loads



  Epoch 1 time: 12.1 min  (726s)
6010/6010 ━━━━━━━━━━━━━━━━━━━━ 726s 109ms/step - class_output_accuracy: 0.3671 - class_output_loss: 6.8612 - class_output_top5_acc: 0.5815 - loss: 7.1050 - nutrition_output_loss: 0.0010 - nutrition_output_mae: 0.0432 - val_class_output_accuracy: 0.1606 - val_class_output_loss: 7.6878 - val_class_output_top5_acc: 0.2216 - val_loss: 7.8497 - val_nutrition_output_loss: 5.2707e-04 - val_nutrition_output_mae: 0.0251 - learning_rate: 0.0010
Epoch 2/60


...

6009/6010 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - class_output_accuracy: 0.3546 - class_output_loss: 5.9160 - class_output_top5_acc: 0.5948 - loss: 6.2177 - nutrition_output_loss: 7.9272e-04 - nutrition_output_mae: 0.0296  Epoch 4 time: 9.8 min  (585s)
6010/6010 ━━━━━━━━━━━━━━━━━━━━ 585s 97ms/step - class_output_accuracy: 0.3546 - class_output_loss: 5.9160 - class_output_top5_acc: 0.5948 - loss: 6.2176 - nutrition_output_loss: 7.9270e-04 - nutrition_output_mae: 0.0296 - val_class_output_accuracy: 0.1243 - val_class_output_loss: 6.9830 - val_class_output_top5_acc: 0.2158 - val_loss: 7.1262 - val_nutrition_output_loss: 5.3062e-04 - val_nutrition_output_mae: 0.0253 - learning_rate: 0.0010
Epoch 5/60
   2/6010 ━━━━━━━━━━━━━━━━━━━━ 8:56 89ms/step - class_output_accuracy: 0.0000e+00 - class_output_loss: 0.0690 - class_output_top5_acc: 0.0000e+00 - loss: 0.2090 - nutrition_output_loss: 1.7200e-07 - nutrition_output_mae: 0.0081  

2026-05-26 21:29:55.615895: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 67109120 bytes after encountering the first element of size 67109120 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


6009/6010 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - class_output_accuracy: 0.3478 - class_output_loss: 5.7506 - class_output_top5_acc: 0.5869 - loss: 6.0263 - nutrition_output_loss: 7.7018e-04 - nutrition_output_mae: 0.0303  Epoch 5 time: 9.8 min  (587s)
6010/6010 ━━━━━━━━━━━━━━━━━━━━ 587s 98ms/step - class_output_accuracy: 0.3478 - class_output_loss: 5.7505 - class_output_top5_acc: 0.5869 - loss: 6.0262 - nutrition_output_loss: 7.7016e-04 - nutrition_output_mae: 0.0303 - val_class_output_accuracy: 0.1380 - val_class_output_loss: 6.6825 - val_class_output_top5_acc: 0.2238 - val_loss: 6.8381 - val_nutrition_output_loss: 5.3208e-04 - val_nutrition_output_mae: 0.0256 - learning_rate: 0.0010
Epoch 6/60
   2/6010 ━━━━━━━━━━━━━━━━━━━━ 8:18 83ms/step - class_output_accuracy: 0.0000e+00 - class_output_loss: 0.0720 - class_output_top5_acc: 0.0000e+00 - loss: 0.2245 - nutrition_output_loss: 1.7000e-07 - nutrition_output_mae: 0.0079  

2026-05-26 21:39:42.362433: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 67109120 bytes after encountering the first element of size 67109120 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


6009/6010 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - class_output_accuracy: 0.3492 - class_output_loss: 5.6429 - class_output_top5_acc: 0.5909 - loss: 5.9341 - nutrition_output_loss: 7.5224e-04 - nutrition_output_mae: 0.0313  Epoch 6 time: 9.7 min  (585s)
6010/6010 ━━━━━━━━━━━━━━━━━━━━ 585s 97ms/step - class_output_accuracy: 0.3492 - class_output_loss: 5.6428 - class_output_top5_acc: 0.5909 - loss: 5.9340 - nutrition_output_loss: 7.5222e-04 - nutrition_output_mae: 0.0313 - val_class_output_accuracy: 0.1399 - val_class_output_loss: 6.7230 - val_class_output_top5_acc: 0.2276 - val_loss: 6.8856 - val_nutrition_output_loss: 5.1765e-04 - val_nutrition_output_mae: 0.0252 - learning_rate: 0.0010

Phase 1 total: 1.01h  (60.8min)
  Train cls acc (last): 34.94%
  Val   cls acc (last): 13.99%
  Best  val cls acc   : 16.06%
  Val   nut MAE (last): 0.0252
Saved phase 1 model.


## 9. Phase 2 — Fine-tuning

In [ ]:
# -- Reload if kernel was restarted --
try:
    model
    base_model
except NameError:
    print(f'Reloading from {SAVE_DIR}/{MODEL_NAME}_final_base.keras ...')
    model      = keras.models.load_model(f'{SAVE_DIR}/{MODEL_NAME}_final_base.keras')
    base_model = next(l for l in model.layers if isinstance(l, tf.keras.Model))
    print(f'Loaded. base_model: {base_model.name}  ({len(base_model.layers)} layers)')

base_model.trainable = True

# -- Freeze everything up to block 6 --
for layer in base_model.layers[:287]:
    layer.trainable = False

# -- Keep BN in inference mode --
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Trainable backbone layers: {trainable_count} / {len(base_model.layers)}')

# -- Reuse tiered dataset (already built) --
try:
    train_ds
    val_ds
except NameError:
    print('Rebuilding datasets...')
    train_ds = build_tiered_dataset(train_paths, train_labels, train_nuts)
    val_ds   = build_val_dataset(val_paths, val_labels, val_nuts)

steps_per_epoch = len(train_paths) // BATCH_SIZE
total_steps     = FINE_TUNE_EPOCHS * steps_per_epoch

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=total_steps,
    alpha=1e-7
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss={
        'class_output':     keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        'nutrition_output': keras.losses.Huber(delta=0.1)
    },
    loss_weights={
        'class_output':     CLS_LOSS_WEIGHT,
        'nutrition_output': NUT_LOSS_WEIGHT
    },
    metrics={
        'class_output': [
            'accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
        ],
        'nutrition_output': ['mae']
    }
)

print('\n=== Phase 2: Fine-tuning ===')
ft_start = time.time()

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[
        ModelCheckpoint(
            f'{SAVE_DIR}/{MODEL_NAME}_best_finetuned.keras',
            save_best_only=True,
            monitor='val_class_output_accuracy'
        ),
        EarlyStopping(
            patience=8,
            restore_best_weights=True,
            monitor='val_class_output_accuracy',
            mode='max'
        ),
        CSVLogger(f'{SAVE_DIR}/{MODEL_NAME}_history_finetuned.log'),
        EpochTimer(),
        LivePlotCallback(SAVE_DIR, MODEL_NAME, phase='phase2'),
    ]
)

ft_total = time.time() - ft_start
h        = history_ft.history

print(f'\nPhase 2 total: {ft_total/3600:.2f}h  ({ft_total/60:.1f}min)')
print(f'  Train cls acc (last): {h["class_output_accuracy"][-1]*100:.2f}%')
print(f'  Val   cls acc (last): {h["val_class_output_accuracy"][-1]*100:.2f}%')
print(f'  Best  val cls acc   : {max(h["val_class_output_accuracy"])*100:.2f}%')
print(f'  Val   nut MAE (last): {h["val_nutrition_output_mae"][-1]:.4f}')

info_path = f'{SAVE_DIR}/{MODEL_NAME}_model_info.txt'
with open(info_path, 'a') as f:
    f.write(f'\n-- Phase 2 Results --\n')
    f.write(f'  Fine-tune from layer   : 287+\n')
    f.write(f'  Training time          : {ft_total/3600:.2f}h\n')
    f.write(f'  Val   cls acc (last)   : {h["val_class_output_accuracy"][-1]*100:.2f}%\n')
    f.write(f'  Best  val cls acc      : {max(h["val_class_output_accuracy"])*100:.2f}%\n')
    f.write(f'  Val nut MAE (last)     : {h["val_nutrition_output_mae"][-1]:.4f}\n')

model.save(f'{SAVE_DIR}/{MODEL_NAME}_final.keras')
print('Saved final model.')


Trainable backbone layers: 180 / 513

=== Phase 2: Fine-tuning ===
Epoch 1/20
6009/6010 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - class_output_accuracy: 0.1148 - class_output_loss: 6.1066 - class_output_top5_acc: 0.2750 - loss: 6.2634 - nutrition_output_loss: 3.8371e-04 - nutrition_output_mae: 0.0240

## 10. Training Curves

In [ ]:
try:
    history_ht
    history_ft
except NameError:
    H = lambda f: type('H', (), {'history': pd.read_csv(f).to_dict('list')})()
    history_ht = H(f'{SAVE_DIR}/{MODEL_NAME}_history_base.log')
    history_ft = H(f'{SAVE_DIR}/{MODEL_NAME}_history_finetuned.log')

def combine(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

n_ht = len(history_ht.history['class_output_accuracy'])
n_ft = len(history_ft.history['class_output_accuracy'])
all_epochs     = list(range(1, n_ht + n_ft + 1))
phase_boundary = n_ht

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'{MODEL_NAME} — Training Curves', fontsize=14)

def plot(ax, key, val_key, title):
    ax.plot(all_epochs, combine(history_ht, history_ft, key),     label='Train')
    ax.plot(all_epochs, combine(history_ht, history_ft, val_key), label='Val')
    ax.axvline(phase_boundary, color='gray', linestyle='--', label='Fine-tune start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()

plot(axes[0,0], 'loss',                       'val_loss',                       'Total Loss')
plot(axes[0,1], 'class_output_accuracy',       'val_class_output_accuracy',       'Top-1 Accuracy')
plot(axes[1,0], 'class_output_top5_acc',       'val_class_output_top5_acc',       'Top-5 Accuracy')
plot(axes[1,1], 'nutrition_output_mae',        'val_nutrition_output_mae',        'Nutrition MAE (normalized)')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/{MODEL_NAME}_training_curves.png', dpi=150)
plt.show()
print(f'Saved → {SAVE_DIR}/{MODEL_NAME}_training_curves.png')

## 11. Evaluation — Classification

In [ ]:
try:
    model
except NameError:
    model = keras.models.load_model(f'{SAVE_DIR}/{MODEL_NAME}_final.keras')

print('Running inference on val set...')
y_true, y_pred, y_pred_probs_list = [], [], []

for images, targets in val_ds:
    cls_preds, _ = model.predict(images, verbose=0)
    y_true.extend(np.argmax(targets['class_output'].numpy(), axis=1))
    y_pred.extend(np.argmax(cls_preds, axis=1))
    y_pred_probs_list.extend(cls_preds)

y_true            = np.array(y_true)
y_pred            = np.array(y_pred)
y_pred_probs_arr  = np.array(y_pred_probs_list)

top1 = np.mean(y_true == y_pred) * 100
top5_hits = [y_true[i] in np.argsort(y_pred_probs_arr[i])[-5:] for i in range(len(y_true))]
top5 = np.mean(top5_hits) * 100

print(f'\n{"="*40}')
print(f'  Top-1 Accuracy : {top1:.2f}%')
print(f'  Top-5 Accuracy : {top5:.2f}%')
print(f'{"="*40}\n')

class_names  = [N_class[i] for i in range(N_CLASSES)]
report       = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
per_class_f1 = {cls: report[cls]['f1-score'] for cls in class_names}

worst20 = sorted(per_class_f1.items(), key=lambda x: x[1])[:20]
best20  = sorted(per_class_f1.items(), key=lambda x: x[1], reverse=True)[:20]

print('20 worst-performing classes (F1):')
for cls, f1 in worst20: print(f'  {cls:<35} F1={f1:.3f}')
print('\n20 best-performing classes (F1):')
for cls, f1 in best20:  print(f'  {cls:<35} F1={f1:.3f}')

## 12. Evaluation — Nutrition Regression

In [ ]:
print('Running nutrition evaluation on val set...')
nut_true_list, nut_pred_list = [], []

for images, targets in val_ds:
    _, nut_preds = model.predict(images, verbose=0)
    nut_true_list.extend(targets['nutrition_output'].numpy())
    nut_pred_list.extend(nut_preds)

nut_true = np.array(nut_true_list)   # normalized [0,1]
nut_pred = np.array(nut_pred_list)

# Denormalize for human-readable MAE
nut_true_raw = denormalize_nutrition(nut_true)
nut_pred_raw = denormalize_nutrition(nut_pred)

print(f'\nNutrition MAE (raw units):')
for i, col in enumerate(NUT_COLS):
    mae = np.mean(np.abs(nut_true_raw[:, i] - nut_pred_raw[:, i]))
    print(f'  {col:<10}  MAE = {mae:.1f}')

## 13. Confusion Matrix (Worst 20 Classes)

In [ ]:
worst_indices = [class_names.index(c) for c, _ in worst20]
mask_worst    = np.isin(y_true, worst_indices)
cm_worst      = confusion_matrix(y_true[mask_worst], y_pred[mask_worst], labels=worst_indices)

plt.figure(figsize=(14, 12))
sns.heatmap(cm_worst, annot=True, fmt='d', cmap='Reds',
            xticklabels=[class_names[i] for i in worst_indices],
            yticklabels=[class_names[i] for i in worst_indices])
plt.title(f'{MODEL_NAME} — Confusion Matrix (20 Worst Classes)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/{MODEL_NAME}_heatmap_worst.svg')
plt.show()

## 14. TFLite Export

In [ ]:
primary_path  = f'{SAVE_DIR}/{MODEL_NAME}_final.keras'
fallback_path = f'{SAVE_DIR}/{MODEL_NAME}_best_finetuned.keras'

model_path = primary_path if os.path.exists(primary_path) else fallback_path
if not os.path.exists(model_path):
    raise FileNotFoundError('No trained model found — run training first')

model = tf.keras.models.load_model(model_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Dynamic range quantization: ~4x smaller, minimal accuracy drop
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_path = f'{SAVE_DIR}/{MODEL_NAME}.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / 1024 / 1024
print(f'TFLite saved → {tflite_path}')
print(f'File size    : {size_mb:.1f} MB')
print()
print('Files to ship with Android app:')
print(f'  {tflite_path}')
print(f'  output/label_map.json')
print(f'  output/class_to_fdcid.json')
print(f'  output/nutrition_stats.json')